# 🏠 Air BNB Data Analysis - NYC Listings

**Comprehensive Analysis of 48,895 Airbnb Listings in New York City**

---

## 📋 Table of Contents
1. [Data Loading & Exploration](#1-data-loading--exploration)
2. [Data Cleaning](#2-data-cleaning)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Price Analysis](#4-price-analysis)
5. [Location Analysis](#5-location-analysis)
6. [Reviews Analysis](#6-reviews-analysis)
7. [Availability Analysis](#7-availability-analysis)
8. [Machine Learning - Price Prediction](#8-machine-learning---price-prediction)
9. [Key Insights & Conclusions](#9-key-insights--conclusions)

---

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully!")

## 1. Data Loading & Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('Airbnb_dataset.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"📝 Total Listings: {len(df):,}")
print(f"📋 Total Columns: {df.shape[1]}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset information
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("\n🔍 Missing Values:\n")
print(missing_df)

## 2. Data Cleaning

In [ ]:
# Handle missing values
df['reviews_per_month'].fillna(0, inplace=True)

# Handle price outliers (keep prices under $1000 for better visualization)
df_clean = df[df['price'] <= 1000].copy()

print(f"✅ Cleaned dataset: {len(df_clean):,} listings")
print(f"🗑️ Removed outliers: {len(df) - len(df_clean):,} listings")

## 3. Exploratory Data Analysis

In [ ]:
# Room type distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
room_counts = df_clean['room_type'].value_counts()
axes[0].pie(room_counts.values, labels=room_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Room Type Distribution', fontsize=14, fontweight='bold')

# Bar chart
room_counts.plot(kind='bar', ax=axes[1], color=['#FF5A5F', '#00A699', '#FC642D'])
axes[1].set_title('Listings Count by Room Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n📊 Room Type Distribution:")
print(room_counts)

In [ ]:
# Neighbourhood group analysis
if 'neighbourhood_group' in df_clean.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    ng_counts = df_clean['neighbourhood_group'].value_counts()
    
    # Bar chart
    ng_counts.plot(kind='bar', ax=axes[0], color='skyblue')
    axes[0].set_title('Listings by Neighbourhood Group', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Neighbourhood Group')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Pie chart
    axes[1].pie(ng_counts.values, labels=ng_counts.index, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Distribution by Neighbourhood Group', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📍 Neighbourhood Group Distribution:")
    print(ng_counts)

## 4. Price Analysis

In [ ]:
# Price statistics
print("💰 Price Statistics:\n")
print(f"Mean Price: ${df_clean['price'].mean():.2f}")
print(f"Median Price: ${df_clean['price'].median():.2f}")
print(f"Std Dev: ${df_clean['price'].std():.2f}")
print(f"Min Price: ${df_clean['price'].min():.2f}")
print(f"Max Price: ${df_clean['price'].max():.2f}")
print(f"\n25th Percentile: ${df_clean['price'].quantile(0.25):.2f}")
print(f"75th Percentile: ${df_clean['price'].quantile(0.75):.2f}")

In [ ]:
# Price distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram
axes[0, 0].hist(df_clean['price'], bins=50, color='#FF5A5F', edgecolor='white')
axes[0, 0].set_title('Price Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Box plot by room type
df_clean.boxplot(column='price', by='room_type', ax=axes[0, 1])
axes[0, 1].set_title('Price Distribution by Room Type', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Room Type')
axes[0, 1].set_ylabel('Price ($)')
plt.sca(axes[0, 1])
plt.xticks(rotation=45)

# Average price by room type
avg_price = df_clean.groupby('room_type')['price'].mean().sort_values()
avg_price.plot(kind='barh', ax=axes[1, 0], color='#00A699')
axes[1, 0].set_title('Average Price by Room Type', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Average Price ($)')
axes[1, 0].set_ylabel('Room Type')

# Price by neighbourhood group
if 'neighbourhood_group' in df_clean.columns:
    avg_price_ng = df_clean.groupby('neighbourhood_group')['price'].mean().sort_values()
    avg_price_ng.plot(kind='barh', ax=axes[1, 1], color='#FC642D')
    axes[1, 1].set_title('Average Price by Neighbourhood Group', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Average Price ($)')
    axes[1, 1].set_ylabel('Neighbourhood Group')

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 most expensive neighbourhoods
top_expensive = df_clean.groupby('neighbourhood')['price'].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
top_expensive.plot(kind='barh', color='coral')
plt.title('Top 15 Most Expensive Neighbourhoods', fontsize=14, fontweight='bold')
plt.xlabel('Average Price ($)')
plt.ylabel('Neighbourhood')
plt.tight_layout()
plt.show()

print("\n💎 Top 10 Most Expensive Neighbourhoods:")
print(top_expensive.head(10))

## 5. Location Analysis

In [ ]:
# Geographic distribution (interactive map)
sample_df = df_clean.sample(min(5000, len(df_clean)))

fig = px.scatter_mapbox(
    sample_df,
    lat='latitude',
    lon='longitude',
    color='price',
    size='price',
    hover_data=['name', 'neighbourhood', 'room_type'],
    title='Geographic Distribution of Listings (Sample: 5000)',
    zoom=10,
    height=600,
    color_continuous_scale='Viridis'
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

In [ ]:
# Top neighbourhoods by listing count
top_neighbourhoods = df_clean['neighbourhood'].value_counts().head(20)

plt.figure(figsize=(12, 6))
top_neighbourhoods.plot(kind='barh', color='steelblue')
plt.title('Top 20 Neighbourhoods by Listing Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Listings')
plt.ylabel('Neighbourhood')
plt.tight_layout()
plt.show()

## 6. Reviews Analysis

In [ ]:
# Review statistics
print("⭐ Reviews Statistics:\n")
print(f"Total Reviews: {df_clean['number_of_reviews'].sum():,}")
print(f"Average Reviews per Listing: {df_clean['number_of_reviews'].mean():.2f}")
print(f"Median Reviews: {df_clean['number_of_reviews'].median():.2f}")
print(f"\nListings with 0 reviews: {(df_clean['number_of_reviews'] == 0).sum():,}")
print(f"Listings with 100+ reviews: {(df_clean['number_of_reviews'] >= 100).sum():,}")

In [ ]:
# Review distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Number of reviews distribution
axes[0].hist(df_clean['number_of_reviews'], bins=50, color='#FF5A5F', edgecolor='white')
axes[0].set_title('Distribution of Number of Reviews', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_ylabel('Frequency')

# Reviews per month distribution
df_with_reviews = df_clean[df_clean['reviews_per_month'] > 0]
axes[1].hist(df_with_reviews['reviews_per_month'], bins=50, color='#00A699', edgecolor='white')
axes[1].set_title('Distribution of Reviews per Month', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Reviews per Month')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Average reviews by room type
avg_reviews = df_clean.groupby('room_type')['number_of_reviews'].mean().sort_values()

plt.figure(figsize=(10, 5))
avg_reviews.plot(kind='barh', color='orange')
plt.title('Average Number of Reviews by Room Type', fontsize=14, fontweight='bold')
plt.xlabel('Average Reviews')
plt.ylabel('Room Type')
plt.tight_layout()
plt.show()

## 7. Availability Analysis

In [ ]:
# Availability statistics
print("📅 Availability Statistics:\n")
print(f"Average Availability: {df_clean['availability_365'].mean():.1f} days/year")
print(f"Median Availability: {df_clean['availability_365'].median():.1f} days/year")
print(f"\nAlways Available (365 days): {(df_clean['availability_365'] == 365).sum():,}")
print(f"Never Available (0 days): {(df_clean['availability_365'] == 0).sum():,}")
print(f"Highly Available (>300 days): {(df_clean['availability_365'] >= 300).sum():,}")

In [ ]:
# Availability distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df_clean['availability_365'], bins=50, color='#00A699', edgecolor='white')
axes[0].set_title('Availability Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Days Available per Year')
axes[0].set_ylabel('Frequency')

# Box plot by room type
df_clean.boxplot(column='availability_365', by='room_type', ax=axes[1])
axes[1].set_title('Availability by Room Type', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Availability (days)')
plt.sca(axes[1])
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 8. Machine Learning - Price Prediction

In [ ]:
# Import ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ ML libraries imported")

In [ ]:
# Prepare data for ML
ml_df = df_clean[['room_type', 'neighbourhood_group', 'latitude', 'longitude', 
                   'minimum_nights', 'number_of_reviews', 'reviews_per_month',
                   'calculated_host_listings_count', 'availability_365', 'price']].copy()

# Remove any remaining missing values
ml_df = ml_df.dropna()

# Encode categorical variables
le_room = LabelEncoder()
ml_df['room_type_encoded'] = le_room.fit_transform(ml_df['room_type'])

if 'neighbourhood_group' in ml_df.columns:
    le_ng = LabelEncoder()
    ml_df['neighbourhood_group_encoded'] = le_ng.fit_transform(ml_df['neighbourhood_group'])
    feature_cols = ['room_type_encoded', 'neighbourhood_group_encoded', 'latitude', 'longitude',
                    'minimum_nights', 'number_of_reviews', 'reviews_per_month',
                    'calculated_host_listings_count', 'availability_365']
else:
    feature_cols = ['room_type_encoded', 'latitude', 'longitude',
                    'minimum_nights', 'number_of_reviews', 'reviews_per_month',
                    'calculated_host_listings_count', 'availability_365']

X = ml_df[feature_cols]
y = ml_df['price']

print(f"✅ Data prepared: {len(ml_df):,} samples")
print(f"Features: {len(feature_cols)}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train):,} samples")
print(f"Testing set: {len(X_test):,} samples")

In [ ]:
# Train Random Forest model
print("🤖 Training Random Forest model...\n")

rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n📊 Model Performance:")
print(f"Mean Absolute Error: ${mae:.2f}")
print(f"Root Mean Squared Error: ${rmse:.2f}")
print(f"R² Score: {r2:.4f}")
print(f"\n✅ Model can predict prices within ±${mae:.2f} on average")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='skyblue')
plt.xlabel('Importance')
plt.title('Feature Importance for Price Prediction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n🎯 Top 5 Most Important Features:")
print(feature_importance.head())

In [ ]:
# Actual vs Predicted prices
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='#FF5A5F')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Actual vs Predicted Prices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Key Insights & Conclusions

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("🎯 KEY INSIGHTS FROM NYC AIRBNB DATA ANALYSIS")
print("="*80)

print(f"\n📊 DATASET OVERVIEW:")
print(f"   • Total Listings: {len(df):,}")
print(f"   • Analysed Listings (cleaned): {len(df_clean):,}")
print(f"   • Unique Neighbourhoods: {df_clean['neighbourhood'].nunique()}")
if 'neighbourhood_group' in df_clean.columns:
    print(f"   • Neighbourhood Groups: {df_clean['neighbourhood_group'].nunique()}")

print(f"\n💰 PRICING INSIGHTS:")
print(f"   • Average Price: ${df_clean['price'].mean():.2f}/night")
print(f"   • Median Price: ${df_clean['price'].median():.2f}/night")
print(f"   • Price Range: ${df_clean['price'].min():.2f} - ${df_clean['price'].max():.2f}")

if 'neighbourhood_group' in df_clean.columns:
    most_expensive_area = df_clean.groupby('neighbourhood_group')['price'].mean().idxmax()
    most_expensive_price = df_clean.groupby('neighbourhood_group')['price'].mean().max()
    print(f"   • Most Expensive Area: {most_expensive_area} (${most_expensive_price:.2f} avg)")

print(f"\n🏠 ROOM TYPE INSIGHTS:")
for room_type, count in df_clean['room_type'].value_counts().items():
    pct = (count / len(df_clean)) * 100
    avg_price = df_clean[df_clean['room_type'] == room_type]['price'].mean()
    print(f"   • {room_type}: {count:,} ({pct:.1f}%) - Avg Price: ${avg_price:.2f}")

print(f"\n⭐ REVIEWS INSIGHTS:")
print(f"   • Total Reviews: {df_clean['number_of_reviews'].sum():,}")
print(f"   • Average Reviews/Listing: {df_clean['number_of_reviews'].mean():.1f}")
no_reviews = (df_clean['number_of_reviews'] == 0).sum()
print(f"   • Listings with No Reviews: {no_reviews:,} ({(no_reviews/len(df_clean)*100):.1f}%)")
high_reviews = (df_clean['number_of_reviews'] >= 100).sum()
print(f"   • Listings with 100+ Reviews: {high_reviews:,} ({(high_reviews/len(df_clean)*100):.1f}%)")

print(f"\n📅 AVAILABILITY INSIGHTS:")
print(f"   • Average Availability: {df_clean['availability_365'].mean():.0f} days/year")
always_avail = (df_clean['availability_365'] == 365).sum()
print(f"   • Always Available: {always_avail:,} ({(always_avail/len(df_clean)*100):.1f}%)")
never_avail = (df_clean['availability_365'] == 0).sum()
print(f"   • Never Available: {never_avail:,} ({(never_avail/len(df_clean)*100):.1f}%)")

print(f"\n🤖 MACHINE LEARNING RESULTS:")
print(f"   • Model: Random Forest Regressor")
print(f"   • Mean Absolute Error: ${mae:.2f}")
print(f"   • R² Score: {r2:.4f}")
print(f"   • Prediction Accuracy: ±${mae:.2f} on average")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)

---

## 🎓 Conclusions

This comprehensive analysis of NYC Airbnb data reveals:

1. **Market Distribution**: The market is dominated by entire homes/apartments and private rooms, with very few shared rooms.

2. **Pricing Patterns**: Significant price variations across neighborhoods, with certain areas commanding premium prices.

3. **Review Activity**: Strong correlation between reviews and pricing, indicating customer satisfaction and listing popularity.

4. **Availability Trends**: Most listings maintain high availability, suggesting professional hosting or investment properties.

5. **Predictive Power**: Machine learning models can predict prices with reasonable accuracy, with location and room type being the most important factors.

### 💡 Recommendations:
- Hosts should focus on location and room type optimization for maximum revenue
- New listings should target underserved neighborhoods with high demand
- Maintaining availability and accumulating positive reviews are crucial for success

---

**Created for**: AI & Data Visualization Internship Portfolio  
**Dataset**: NYC Airbnb Open Data (48,895 listings)  
**Date**: October 2025